# 🪙 암호화폐 리스크 패리티(Risk Parity) 자산배분

암호화폐 시장은 자산 간 변동성 차이가 매우 큽니다. 단순 1/N 균등 배분이나 시가총액 가중 배분을 사용하면, 변동성이 큰 알트코인에 전체 계좌의 위험이 집중될 수 있습니다.

이 노트북에서는 **Risk Budgeting (Risk Parity / Equal Risk Contribution)** 기법을 사용하여, **각 암호화폐가 포트폴리오 전체 위험에 기여하는 비중을 동일하게 맞추는 최적 자산배분**을 계산합니다.

In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트 경로를 sys.path에 추가하여 scripts 모듈 탐색 보장
project_root = str(Path.cwd().parent.resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd

from skfolio import RiskMeasure
from skfolio.optimization import RiskBudgeting, MeanVariance, ObjectiveFunction
from skfolio.preprocessing import prices_to_returns
from scripts.crypto_portfolio_optimizer import load_from_feather_dir, generate_synthetic_crypto_data, find_freqtrade_data_dirs

print("모듈 로드 완료!")


## 1. Freqtrade 데이터 또는 크립토 시세 불러오기

저장소 내 Freqtrade 다운로드 데이터(`feather` 파일)가 있으면 자동으로 불러오고, 없으면 실전급 합성 시세 데이터를 사용합니다.

In [ ]:
data_dirs = find_freqtrade_data_dirs()
prices = pd.DataFrame()

if data_dirs:
    print(f"Freqtrade 데이터 디렉토리 발견: {data_dirs[0]}")
    prices = load_from_feather_dir(data_dirs[0], timeframe="15m")

if prices.empty:
    print("합성 크립토 데이터셋(BTC, ETH, SOL, XRP)을 생성합니다.")
    prices = generate_synthetic_crypto_data(periods=2000)

returns = prices_to_returns(prices)
print(f"분석 대상 자산군: {list(returns.columns)}")
print(f"총 캔들 개수: {len(returns)}")

## 2. 리스크 패리티(Risk Parity) vs 전통 모델 최적화 비교

In [ ]:
# 1) Risk Parity (위험 균등 기여)
model_rp = RiskBudgeting(risk_measure=RiskMeasure.VARIANCE)
model_rp.fit(returns)

# 2) Maximum Sharpe Ratio (샤프 최대화)
model_sharpe = MeanVariance(
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    risk_measure=RiskMeasure.VARIANCE,
)
model_sharpe.fit(returns)

# 3) Minimum Semi-Variance (하방 낙폭 최소화)
model_semi_var = MeanVariance(
    objective_function=ObjectiveFunction.MINIMIZE_RISK,
    risk_measure=RiskMeasure.SEMI_VARIANCE,
)
model_semi_var.fit(returns)

weights_summary = pd.DataFrame({
    "Risk Parity": model_rp.weights_,
    "Max Sharpe": model_sharpe.weights_,
    "Min Semi-Variance": model_semi_var.weights_,
}, index=returns.columns)

(weights_summary * 100).round(2).astype(str) + "%"

## 3. 결과 해석

- **Risk Parity**: 변동성이 상대적으로 작은 메이저(BTC 등)에 더 높은 비중을 두고, 변동성이 큰 자산에는 적은 비중을 두어 자산군 전체의 위험 기여도를 균등하게 배분합니다.
- **Min Semi-Variance**: 일반적인 가격 변동(상승)은 제한하지 않고, 급락/폭락(하방 변동성)만 집중적으로 회피하는 최적 비중을 산출합니다.
- Freqtrade 봇에서 각 페어의 `stake_amount`(투자금)를 배분할 때 이 최적 비중을 적용할 수 있습니다.